In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('RDD Basics').getOrCreate()

sc = spark.sparkContext

print("Spark Ready")

In [ ]:
data = [10,20,30,40]

rdd = sc.parallelize(data)
print(rdd)



ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:297


In [ ]:
rdd.collect()

[10, 20, 30, 40]

In [ ]:
names = ["Steve", "Jerry", "John", "Suba"]

rdd = sc.parallelize(names)
rdd.collect()

['Steve', 'Jerry', 'John', 'Suba']

In [ ]:
rdd.getNumPartitions()

2

In [ ]:
rdd = sc.parallelize(names, 4)
rdd.getNumPartitions()

4

In [ ]:
rdd = sc.parallelize([10,20,30,50,70], 4)
rdd.glom().collect()

[[10], [20], [30], [50, 70]]

In [ ]:
sample = sc.parallelize([1,2,3,4,5,6], 3)
sample.glom().collect()

[[1, 2], [3, 4], [5, 6]]

Lazy Evaluation - Spark executes only after it sees a action...
It doesc't execute any transformations unnecessarily...

In [ ]:
rdd = sc.parallelize([1,2,3,4,5])

mapped_rdd = rdd.map(lambda x: x*2)
filtered_rdd = mapped_rdd.filter(lambda x: x > 5)

print("Nothing executed yet as map and filter are transformations")

Nothing executed yet as map and filter are transformations


"Execution of previous block starts only after colect() action"

In [ ]:
filtered_rdd.collect()

[6, 8, 10]

RDD operations: two categories

1. Transformations

These create a new RDD.

Examples:

•	map

•	filter

•	flatMap

•	distinct

•	union

•	intersection

•	groupByKey

•	reduceByKey

--------------------------------------------

2. Actions

These return a result to the driver.

Examples:

•	collect

•	count

•	first

•	take

•	reduce


**TRANSFORMATIONS**

In [ ]:
rdd = sc.parallelize([10,20,30,40])

squared_rdd = rdd.map(lambda x: x**2)

squared_rdd.collect()

[100, 400, 900, 1600]

In [ ]:
rdd = sc.parallelize(["STeve", "Davi", "Jabin"])

upper_names = rdd.map(lambda x: x.upper())

upper_names.collect()

['STEVE', 'DAVI', 'JABIN']

In [ ]:
rdd = sc.parallelize([10,20,30,35,40])

filtered_rdd = rdd.map(lambda x: x%2 == 0)

filtered_rdd.collect()

[True, True, True, False, True]

In [ ]:
rdd = sc.parallelize([10,20,30,35,40])

filtered_rdd = rdd.filter(lambda x: x%2 == 0)

filtered_rdd.collect()

[10, 20, 30, 40]

In [ ]:
rdd = sc.parallelize(["apple","banana", "mango"])

long_rdd = rdd.filter(lambda x: len(x) >= 6)

long_rdd.collect()

['banana']

FlatMap - To iterate multiple elements ... Used in counting number of words for ML modelling

In [ ]:
lines = sc.parallelize([
    "python spark hadoop",
    "spark mysql mssql"
])

words = lines.flatMap(lambda x: x.split(" "))

words.collect()

['python', 'spark', 'hadoop', 'spark', 'mysql', 'mssql']

In [ ]:
lines = sc.parallelize([
    "python spark hadoop",
    "spark mysql mssql"
])

words = lines.map(lambda x: x.split(" "))

words.collect()

[['python', 'spark', 'hadoop'], ['spark', 'mysql', 'mssql']]

DISTINCT()

In [ ]:
rdd = sc.parallelize([10,20,30, 40, 20, 10, 20, 30, 20])

rdd.distinct().collect()

[10, 20, 30, 40]

UNION

In [ ]:
rdd1 = sc.parallelize([1,2,3,4])
rdd2 = sc.parallelize([5,6,7,8])

rdd1.union(rdd2).collect()

[1, 2, 3, 4, 5, 6, 7, 8]

INTERSECTION

In [ ]:
rdd1 = sc.parallelize([1,2,3,4])
rdd2 = sc.parallelize([3,4,5,6,7,8])

rdd1.intersection(rdd2).collect()

[4, 3]

SUBTRACT

In [ ]:
rdd1 = sc.parallelize([1,2,3,4])
rdd2 = sc.parallelize([1,5,6,7,8])

rdd1.subtract(rdd2).collect()

[4, 2, 3]

**ACTIONS**

In [ ]:
rdd = sc.parallelize([10,20,30])
rdd.collect()

[10, 20, 30]

In [ ]:
rdd = sc.parallelize([10,20,30])
rdd.count()

3

In [ ]:
rdd = sc.parallelize([10,20,30])
rdd.first()

10

In [ ]:
rdd = sc.parallelize([10,20,30,40])
rdd.take(2)

[10, 20]

In [ ]:
rdd = sc.parallelize([10,20,30,40])
rdd.reduce(lambda x, y : x+y)

100

Key Value Pairs

In [ ]:
data = [
    ("IT", 75000),
    ("Finance", 100000),
    ("IT", 80000),
    ("HR", 50000),
    ("HR", 90000)
]

rdd = sc.parallelize(data)
rdd.collect()

[('IT', 75000),
 ('Finance', 100000),
 ('IT', 80000),
 ('HR', 50000),
 ('HR', 90000)]

In [ ]:
rdd.reduceByKey(lambda x, y: x + y).collect()

[('IT', 155000), ('Finance', 100000), ('HR', 140000)]

In [ ]:
grouped = rdd.groupByKey()

[(k, list(v)) for k, v in grouped.collect()]

[('IT', [75000, 80000]), ('Finance', [100000]), ('HR', [50000, 90000])]

In [ ]:
data = sc.parallelize([
    ("Steve", 100),
    ("Davi", 95),
    ("Jabin", 90)
])
data.mapValues(lambda x: int(x*1.1)).collect()


[('Steve', 110), ('Davi', 104), ('Jabin', 99)]

In [ ]:
rdd = sc.parallelize([43,23,4,10,20,30,40])

rdd.sortBy(lambda x: x).collect()

[4, 10, 20, 23, 30, 40, 43]

In [ ]:
lines = sc.parallelize([
    "python spark hadoop",
    "spark mysql mssql",
    "hadoop mysql spark"
])

word_counts = (
    lines
    .flatMap(lambda line: line.split(" "))
    .map(lambda word: (word, 1))
    .reduceByKey( lambda a, b: a + b)
)

word_counts.collect()

[('python', 1), ('hadoop', 2), ('spark', 3), ('mysql', 2), ('mssql', 1)]

In [ ]:
salary = sc.parallelize([
    ("Steve", 300000),
    ("Tom", 50000),
    ("Sam", 100000),
    ("Davi", 200000)
])

high_salary = salary.filter(lambda x: x[1] > 100000)
high_salary.collect()

[('Steve', 300000), ('Davi', 200000)]